In [3]:
import vibechecker as vc
import yaml
import pandas as pd
import numpy as np
import plotly.express as px
import h5py
import time

In [ ]:
with open('DEVDATA/rotorkit_1800rpm_2025-12-30_15-03-02.csv', 'r') as f:
    raw_data = pd.read_csv(f).to_numpy().flatten()

with open('DEVDATA/rotorkit_1800rpm_2025-12-30_15-03-02.yaml', 'r') as f:
    meta = yaml.load(f, Loader=yaml.SafeLoader)

samp = pd.Series(raw_data)
meta['data'] = raw_data

outfile = 'DEVDATA/rotorkit_1800rpm_2025-12-30_15-03-02.h5'
with h5py.File(outfile, 'w') as h5:
    dataset = h5.create_dataset('sample', data = meta )

time.sleep(1)

with h5py.File(outfile, 'r') as h5:
    data = h5['sample'][:]

display(data)

# with pd.HDFStore() as store:
#     store.put('sample', samp, format='fixed')
#     store.put('meta', pd.DataFrame(meta), format='table')
#     # store.get_storer('sample').attrs.metadata = samp.attrs

{'integration': 'acceleration',
 'raw_unit': 'g',
 'samplerate': 8000,
 'status': {'state': [None, {'_flags': 0}]},
 'timestamp': 994701.1617420999}

TypeError: Cannot serialize the column [status]
because its data contents are not [string] but [mixed] object dtype

In [26]:
config = vc.AcquisitionSettings(len(samp.raw_data), samp.samplerate, units='g', fft_integration=False)
vtime, vaccel = vc.sample_accel(samp, config)
vfreq, vpsd, peaks = vc.sample_spectrum(samp, config)
max(vpsd)

np.float64(0.2503393013744402)

In [ ]:
fm = 200000
df = 5

requires_fs = 2*fm
try:
    fs = next(filter(lambda x: x > requires_fs, vc.SAMPLERATES))
except StopIteration:
    fs = vc.SAMPLERATES[-1]
requires_ns = int(fs/df)
display(vc.SAMPLERATES, vc.BLOCKSIZES)
display(fs,requires_ns)

In [ ]:
# Launch GUI
app = vc.GUI()
app.run()

In [ ]:
# Check method to hotwap sd sensors
vc.sounddevice._terminate()
vc.sounddevice._initialize()

# List devices
devs = vc.VibeSensor.find()

print('ID\tNAME')
for s in devs:
    print(f'{s.device_id}:\t{s.model_name}')

In [ ]:
# Generate and visualize simulated data
dev = vc.VibeSensor.find()

if len(dev)>1:
    sensor = dev[1]
else:
    sensor = dev[0]

print(sensor)

config = vc.AcquisitionSettings(4096,8000)

config.ensure_maxfreq(1000)
config.ensure_binsize(2.0)

vibr = vc.DataCollector(sensor=sensor, config=config)
samp = vibr.collect_sample()
vis = vibr.visualize_init(samp)

vibr.disconnect_sensor()

In [ ]:
import scipy.signal
import matplotlib.pyplot as plt

freq, psd, peak = vc.sample_spectrum(samp, config)

peak, props = scipy.signal.find_peaks(psd)

npeak = 13
peak = sorted(peak, key=lambda x: psd[x], reverse = True)[:npeak]

plt.plot(freq,psd)
plt.semilogy(freq[peak], psd[peak], 'x')
plt.show()

scipy.signal.peak_prominences(psd, peak)


In [ ]:
scipy.signal.peak_widths(psd, peak, )

In [ ]:
import pandas as pd
import numpy as np

x = np.random.rand(10)
df = pd.DataFrame({'x': x, 'y': np.sin(x)})

df.head(n=20)

In [ ]:
import dearpygui.dearpygui as dpg

fft_data_x = list(range(100))
fft_data_y = [i**2 for i in fft_data_x]

dpg.create_context()

def update_crosshair(sender, app_data):
    plot = dpg.get_plot_mouse_pos()
    x, y = plot[0], plot[1]

    # Check if mouse is within plot bounds (get_plot_mouse_pos returns extreme values when outside)
    if x > 1e6 or x < -1e6:
        if dpg.does_item_exist("v_line"):
            dpg.delete_item("v_line")
        if dpg.does_item_exist("tooltip"):
            dpg.delete_item("tooltip")
        return

    # Find closest index
    idx = min(range(len(fft_data_x)), key=lambda i: abs(fft_data_x[i] - x))
    y_value = fft_data_y[idx]

    # Update or create crosshair
    if dpg.does_item_exist("v_line"):
        dpg.configure_item("v_line", p1=[fft_data_x[idx], 0], p2=[fft_data_x[idx], y_value])
    else:
        dpg.draw_line([fft_data_x[idx], 0], [fft_data_x[idx], y_value], 
                      color=[255, 0, 0, 255], thickness=2, parent="plot", tag="v_line")

    if dpg.does_item_exist("tooltip"):
        dpg.configure_item("tooltip", default_value=f"Value: {y_value}", pos=[fft_data_x[idx] + 10, y_value + 10])
    else:
        dpg.draw_text(pos=[fft_data_x[idx] + 10, y_value + 10], text=f"Value: {y_value}", 
                      color=[255, 255, 0, 255], parent="plot", tag="tooltip")

# Mouse move handler
with dpg.handler_registry():
    dpg.add_mouse_move_handler(callback=update_crosshair)

with dpg.window(label="Plot Window"):
    with dpg.plot(label="FFT Plot", height=400, width=600, tag="plot"):
        dpg.add_plot_axis(dpg.mvXAxis, label="X")
        dpg.add_plot_axis(dpg.mvYAxis, label="Y")
        dpg.add_line_series(fft_data_x, fft_data_y, label="FFT Data", parent=dpg.last_item(), tag="fft_data")

dpg.create_viewport()
dpg.setup_dearpygui()
dpg.show_viewport()
dpg.start_dearpygui()
dpg.destroy_context()     